# TreeIG vs TreeSHAP

This notebook gives a minimal comparison of TreeIG and TreeSHAP for a fitted tree ensemble.

TreeIG computes exact Integrated Gradients for tree models along a straight-line path from a chosen baseline input `x0` to an observation `x`. The attributions decompose the model's endpoint change:

```text
sum_j phi_j(x; x0) = f(x) - f(x0)
```

TreeSHAP computes Shapley-value attributions for tree models. SHAP and IG answer different attribution questions. TreeIG is baseline/path based. TreeSHAP is coalition/feature-removal based. The two methods can therefore produce different attributions even when both are internally consistent.


## Setup

The notebook assumes that `treeig`, `xgboost`, `scikit-learn`, `shap`, `numpy`, `pandas`, and `matplotlib` are installed.

For a fresh environment, one possible setup is:

```bash
python -m pip install treeig xgboost scikit-learn shap pandas matplotlib
```


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import treeig
from treeig import TreeIG

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

warnings.filterwarnings("ignore", category=UserWarning)

print("treeig", treeig.__version__)
print("shap available:", HAS_SHAP)


## Simulate a nonlinear regression problem

The data-generating process includes nonlinearities and interactions so that a tree ensemble has meaningful structure to explain.


In [ ]:
rng = np.random.default_rng(2026)

n = 2500
p = 6
feature_names = [f"x{j}" for j in range(p)]

X = rng.normal(size=(n, p))
y = (
    1.5 * X[:, 0]
    - 1.0 * X[:, 1] ** 2
    + 0.8 * X[:, 2] * X[:, 3]
    + np.sin(1.5 * X[:, 4])
    - 0.4 * np.maximum(X[:, 5], 0.0)
    + 0.15 * rng.normal(size=n)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

print(X_train_df.shape, X_test_df.shape)


## Fit an XGBoost model

TreeIG attributes the fitted model, not the data-generating process. The model is deliberately small so the notebook runs quickly.


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=80,
    max_depth=3,
    learning_rate=0.06,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=0,
    n_jobs=1,
    verbosity=0,
)

model.fit(X_train_df, y_train)

pred = model.predict(X_test_df)
print("test R^2:", round(r2_score(y_test, pred), 4))


## Compute TreeIG attributions

We use the mean training input as the baseline. For each test observation, TreeIG decomposes the prediction difference from this baseline.


In [ ]:
x0 = X_train_df.mean(axis=0).to_numpy()
X_batch = X_test_df.iloc[:200].to_numpy()

ig = TreeIG(model, baseline=x0).warmup(X_batch[:5])

phi_ig, infos, summary = ig.explain(X_batch)

print("TreeIG shape:", phi_ig.shape)
print("max absolute residual:", summary["max_abs_residual"])
print("mean events:", summary["mean_events"])


## Verify TreeIG completeness

The defining check is that the row sum of the attribution matrix equals the prediction difference from the baseline.


In [ ]:
f_x = model.predict(X_batch.astype(np.float32))
f_0 = model.predict(x0.reshape(1, -1).astype(np.float32))[0]
endpoint_delta = f_x - f_0

check = pd.DataFrame(
    {
        "prediction_delta": endpoint_delta,
        "treeig_sum": phi_ig.sum(axis=1),
        "residual": phi_ig.sum(axis=1) - endpoint_delta,
    }
)

print(check.head())
print("max abs residual:", np.max(np.abs(check["residual"])))


## Compute TreeSHAP attributions

TreeSHAP uses a different attribution principle. Depending on SHAP version and settings, the SHAP baseline is represented by the explainer's expected value. We compute SHAP values for the same batch and compare them with TreeIG.


In [ ]:
if HAS_SHAP:
    shap_explainer = shap.TreeExplainer(model)
    shap_values_raw = shap_explainer.shap_values(X_batch)

    if isinstance(shap_values_raw, list):
        # Binary or multiclass APIs may return a list. This notebook uses regression.
        phi_shap = np.asarray(shap_values_raw[0], dtype=float)
    else:
        phi_shap = np.asarray(shap_values_raw, dtype=float)

    expected_value = shap_explainer.expected_value
    if isinstance(expected_value, (list, tuple, np.ndarray)):
        expected_value = np.asarray(expected_value).ravel()[0]

    shap_sum = phi_shap.sum(axis=1)
    print("TreeSHAP shape:", phi_shap.shape)
    print("SHAP expected value:", float(expected_value))
    print("first five SHAP sums:", shap_sum[:5])
else:
    phi_shap = None
    expected_value = None
    print("Install shap to run this section: python -m pip install shap")


## Compare one observation

The two methods need not agree feature by feature. TreeIG explains movement from the selected baseline along a path. TreeSHAP explains the model output through a Shapley-value feature-allocation rule.


In [ ]:
obs = 0

comparison = pd.DataFrame(
    {
        "feature": feature_names,
        "x0": x0,
        "x": X_batch[obs],
        "TreeIG": phi_ig[obs],
    }
)

if phi_shap is not None:
    comparison["TreeSHAP"] = phi_shap[obs]
    comparison["abs_difference"] = np.abs(comparison["TreeIG"] - comparison["TreeSHAP"])

comparison = comparison.sort_values("TreeIG", key=np.abs, ascending=False)
comparison


In [ ]:
plot_df = comparison.set_index("feature")
cols = ["TreeIG"] + (["TreeSHAP"] if phi_shap is not None else [])

ax = plot_df[cols].plot(kind="bar", figsize=(9, 4))
ax.axhline(0.0, linewidth=1)
ax.set_title("Feature attributions for one observation")
ax.set_ylabel("attribution")
plt.tight_layout()
plt.show()


## Compare average absolute attributions

Average absolute attributions give a simple global summary over the selected batch. This is not a formal feature-importance theorem; it is just a useful descriptive comparison.


In [ ]:
global_df = pd.DataFrame(
    {
        "feature": feature_names,
        "TreeIG_mean_abs": np.mean(np.abs(phi_ig), axis=0),
    }
)

if phi_shap is not None:
    global_df["TreeSHAP_mean_abs"] = np.mean(np.abs(phi_shap), axis=0)

global_df = global_df.sort_values("TreeIG_mean_abs", ascending=False)
global_df


In [ ]:
plot_global = global_df.set_index("feature")
cols = ["TreeIG_mean_abs"] + (["TreeSHAP_mean_abs"] if phi_shap is not None else [])

ax = plot_global[cols].plot(kind="bar", figsize=(9, 4))
ax.set_title("Mean absolute attributions over the batch")
ax.set_ylabel("mean absolute attribution")
plt.tight_layout()
plt.show()


## Simple runtime comparison

The timing below is intentionally informal. It is meant to show the order of magnitude for this small example, not to serve as a careful benchmark.


In [ ]:
def time_call(fn, n_repeat=5):
    times = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return np.array(times)

# Warm up once.
_ = ig.attribute(X_batch)
if HAS_SHAP:
    _ = shap_explainer.shap_values(X_batch)

treeig_times = time_call(lambda: ig.attribute(X_batch), n_repeat=5)

print("TreeIG seconds:", treeig_times)
print("TreeIG median seconds:", np.median(treeig_times))

if HAS_SHAP:
    shap_times = time_call(lambda: shap_explainer.shap_values(X_batch), n_repeat=5)
    print("TreeSHAP seconds:", shap_times)
    print("TreeSHAP median seconds:", np.median(shap_times))


## Takeaways

1. TreeIG gives an exact additive decomposition of `f(x) - f(x0)` for tree ensembles.
2. The baseline is an explicit part of the TreeIG question.
3. TreeSHAP and TreeIG are both useful, but they answer different attribution questions.
4. Differences between TreeIG and TreeSHAP attributions are therefore expected, not necessarily a sign that either method is wrong.
